## CMA-ES for CartPole POMDP (Assignment)
### Christian Igel, 2026

If you have suggestions for improvement, [let me know](mailto:igel@diku.dk).

You may need the following packages:

``pip install gymnasium[classic-control]``

``python -m pip install cma``

In [1]:
import gymnasium as gym  # Defines RL environments

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output  # For inline visualization

import torch
import torch.nn as nn
import torch.nn.functional as F

import cma

Define RL task:

In [2]:
# Define task
env = gym.make('CartPole-v1')
state_space_dimension  = 2 # env.observation_space.shape[0]
action_space_dimension = 1  

Define the policy network:

In [3]:
# Model definition
class PolicyRNN(nn.Module):
    def __init__(self, state_size=2, action_size=1, hidden_size=5, bias=False):
        super(PolicyRNN, self).__init__()
        self.hidden_size = hidden_size
        self.rnn = nn.RNN(state_space_dimension, hidden_size=hidden_size, num_layers=1, nonlinearity='tanh', 
                          bias=bias, batch_first=False, dropout=0.0, bidirectional=False)
        self.fc = nn.Linear(hidden_size, action_size, bias=bias)

    def zero_state_nobatch(self):
        return torch.zeros(1, self.hidden_size)
    
    def forward(self, x, rnn_state):
        x, new_rnn_state = self.rnn(x, rnn_state)
        x = self.fc(x)
        return x, new_rnn_state
    
policy_net = PolicyRNN(state_space_dimension, action_space_dimension, hidden_size=2, bias=False)

Print network structure and compute number of parameters:

In [4]:
print(policy_net)
d = sum(
	param.numel() for param in policy_net.parameters()
)
print("Number of parameters:", d)

PolicyRNN(
  (rnn): RNN(2, 2, bias=False)
  (fc): Linear(in_features=2, out_features=1, bias=False)
)
Number of parameters: 10


We turn the problem into solving a POMDP by ignoring the velocities:

In [5]:
state, _  = env.reset()
print(state, state[[0,2]])

[ 0.03569175 -0.01519395  0.00551778 -0.00728842] [0.03569175 0.00551778]


Helper function for visualization:

In [6]:
def visualize_policy(policy_net):
    env_render = gym.make('CartPole-v1', render_mode='rgb_array')
    state, _ = env_render.reset()  # Forget about previous episode
    state_tensor = torch.Tensor( state[[0,2]].reshape((1, state_space_dimension)) )
    steps = 0
    rnn_state = policy_net.zero_state_nobatch()
    while True:
        out, rnn_state = policy_net(state_tensor, rnn_state)
        a = int(out > 0)
        state, reward, terminated, truncated, _ = env_render.step(a)  # Simulate pole
        steps+=1
        state_tensor = torch.Tensor( state[[0,2]].reshape((1, state_space_dimension)) )
        clear_output(wait=True)
        plt.imshow(env_render.render())
        plt.show()
        print("step:", steps)
        if(terminated or truncated): 
            break
    env_render.close()
    return

Now we define the objective/reward function. 
When the task is solved the functions returns -1000.
One successful trial is sufficient.

In [7]:
def fitness_cart_pole(x, nn, env):
    '''
    Returns negative accumulated reward for single pole, fully environment.

    Parameters:
        x: Parameter vector encoding the weights.
        nn: Parameterized model.
        env: Environment ('CartPole-v?').
    '''
    torch.nn.utils.vector_to_parameters(torch.Tensor(x), nn.parameters())  # Set the policy parameters
    
    state, _ = env.reset()  # Forget about previous episode
    state_tensor = torch.Tensor(state[[0,2]].reshape((1, state_space_dimension)) )
    rnn_state = nn.zero_state_nobatch()    
    R = 0  # Accumulated reward
    while True:
        out, rnn_state = nn(state_tensor, rnn_state)
        a = int(out > 0)
        state, reward, terminated, truncated, _ = env.step(a)  # Simulate pole
        state_tensor = torch.Tensor( state[[0,2]].reshape((1, state_space_dimension)) )
        R += reward  # Accumulate 
        if truncated:
            return -1000  # Episode ended, final goal reached, we consider minimization
        if terminated:
            return -R  # Episode ended, we consider minimization
    return -R  # Never reached  

Do the learning:

In [8]:
# Generate initial search point and initial hidden RNN states
initial_weights = np.random.normal(0, 0.1, d)  # Random parameters for initial policy, d denotes the number of weights
initial_sigma = .1 # Initial global step-size sigma

# Do the optimization
res = cma.fmin(fitness_cart_pole,  # Objective function
          initial_weights,  # Initial search point
          initial_sigma,  # Initial global step-size sigma
          args=([policy_net, env]),  # Arguments passed to the fitness function
          options={'ftarget': -999.9, 'tolflatfitness':1e5, 'eval_final_mean':False})
env.close()

# Set the policy parameters to the final solution
torch.nn.utils.vector_to_parameters(torch.Tensor(res[0]), policy_net.parameters())  

print("best solution found after", res[2], "evaluations")

(5_w,10)-aCMA-ES (mu_w=3.2,w_1=45%) in dimension 10 (seed=676542, Wed Mar  4 19:51:09 2026)
Iterat #Fevals   function value  axis ratio  sigma  min&max std  t[m:s]
    1     10 -3.600000000000000e+01 1.0e+00 9.64e-02  9e-02  1e-01 0:00.1
    2     20 -3.400000000000000e+01 1.2e+00 9.25e-02  9e-02  9e-02 0:00.2
    3     30 -5.100000000000000e+01 1.2e+00 9.40e-02  9e-02  1e-01 0:00.2
   16    160 -1.000000000000000e+03 2.0e+00 1.28e-01  1e-01  1e-01 0:01.5
termination on {'ftarget': -999.9} (Wed Mar  4 19:51:10 2026)
final/bestever f-value = -1.000000e+03 -1.000000e+03 after 160/154 evaluations
incumbent solution: [ 0.30617059 -0.04508384  0.02234983  0.55616617 -0.08793343 -0.31049431
  0.27488561 -0.42695882 ...]
std deviations: [0.12512866 0.1357765  0.12643898 0.13274552 0.1239606  0.13280463
 0.12280182 0.12357796 ...]
best solution found after 154 evaluations


In [9]:
r_bias = []
r_bias1 = []
r_non = []
r_non1 = []
for b in [True, False]:
    results = []
    results1 = []
    for i in range(10):
        np.random.seed(i)
        policy_net = PolicyRNN(state_space_dimension, action_space_dimension, hidden_size=2, bias=b)
        d = sum(param.numel() for param in policy_net.parameters())
        initial_weights = np.random.normal(0, 0.1, d)  # Random parameters for initial policy, d denotes the number of weights
        initial_sigma = .1 # Initial global step-size sigma

        # Do the optimization
        res = cma.fmin(fitness_cart_pole,  # Objective function
                  initial_weights,  # Initial search point
                  initial_sigma,  # Initial global step-size sigma
                  args=([policy_net, env]),  # Arguments passed to the fitness function
                  options={'ftarget': -999.9, 'tolflatfitness':1e5, 'eval_final_mean':False})
        env.close()

        # Set the policy parameters to the final solution
        torch.nn.utils.vector_to_parameters(torch.Tensor(res[0]), policy_net.parameters())

        results.append(res[2])
        results1.append(res[1])
        print(i)

    print(f"Best solution found after, {np.mean(results)} in average for 10 trials, evaluations, for bias = {b}")
    if b == True:
        r_bias = results
        r_bias1 = results
    else:
        r_non = results
        r_non1 = results


(6_w,12)-aCMA-ES (mu_w=3.7,w_1=40%) in dimension 15 (seed=818715, Wed Mar  4 19:51:10 2026)
Iterat #Fevals   function value  axis ratio  sigma  min&max std  t[m:s]
    1     12 -1.000000000000000e+01 1.0e+00 9.62e-02  9e-02  1e-01 0:00.0
    2     24 -1.000000000000000e+01 1.1e+00 9.92e-02  1e-01  1e-01 0:00.1
    3     36 -1.000000000000000e+01 1.2e+00 1.05e-01  1e-01  1e-01 0:00.1
   69    828 -5.900000000000000e+01 3.9e+00 1.53e-01  9e-02  2e-01 0:03.1
  100   1200 -5.000000000000000e+01 4.8e+00 1.82e-01  1e-01  2e-01 0:06.1
  150   1800 -6.900000000000000e+01 5.6e+00 1.52e-01  8e-02  2e-01 0:11.2
  200   2400 -7.000000000000000e+01 7.8e+00 1.05e-01  6e-02  1e-01 0:16.2
  269   3228 -6.700000000000000e+01 9.3e+00 6.48e-02  4e-02  8e-02 0:23.2
  300   3600 -6.300000000000000e+01 9.3e+00 6.58e-02  4e-02  8e-02 0:26.5
  385   4620 -9.400000000000000e+01 1.2e+01 4.46e-02  2e-02  6e-02 0:35.6
  400   4800 -8.400000000000000e+01 1.3e+01 4.63e-02  2e-02  6e-02 0:37.5
  500   6000 -8.200000

Visualize solution:

In [10]:
visualize_policy(policy_net)

DependencyNotInstalled: pygame is not installed, run `pip install "gymnasium[classic-control]"`

Make a plot with an inset image:

In [ ]:
# Load previous optimization results 
logger = res[-1]  # the CMADataLogger
logger.load();    # by "default" data are on disk

# Render solution and take a still image
env_render = gym.make('CartPole-v1', render_mode='rgb_array')
state, _ = env_render.reset()  # Forget about previous episode
img = env_render.render()

# Make plot
fig, ax = plt.subplots()
ax.plot(logger.f[:,1], -logger.f[:,5], 'bo-')  # plot f versus iteration, see file header
ax.set_xlabel("function evaluations")
ax.set_ylabel("return")
ax.grid()
ax.set_yscale('log')

# Define the position and size parameters
image_xaxis = 0.2
image_yaxis = 0.375
image_width = 0.5
image_height = 0.4  # Same as width since our logo is a square

# Define the position for the image axes
ax_image = fig.add_axes([image_xaxis,
                         image_yaxis,
                         image_width,
                         image_height]
                       );

# Display the image
ax_image.imshow(img)
ax_image.axis('on');
ax_image.set_xticks([])  # Remove x-axis ticks
ax_image.set_yticks([])  # Remove y-axis ticks
fig.savefig("CMA-Pole-POMDP.pdf")